# T12 — Quantile Models: Q-MLP / Q-RNN / Q-LSTM / Q-GRU / Q-Transformer

Quantile models predict THREE values simultaneously:
- **Q10** (10th percentile) = optimistic RUL (engine will last at least this long)
- **Q50** (50th percentile) = median RUL (best single estimate)
- **Q90** (90th percentile) = pessimistic RUL (conservative safety bound)

Key difference from standard DL (T11):
- Loss function: **Pinball loss** instead of MSE
- Output head: **3 neurons** instead of 1
- Evaluation: Q50 used for RMSE/NASA comparison with classical models

Why quantile matters for predictive maintenance:
- Q10 = 'worst case' — use this for safety-critical decisions
- Q90 = 'best case' — use this for maintenance scheduling
- Interval width = model uncertainty — wide = model unsure about this engine

In [ ]:
import sys, os
from pathlib import Path

ROOT = Path(os.getcwd()).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from src.evaluation.metrics import evaluate

# ── Constants ─────────────────────────────────────────────────────────
WINDOW_SIZE = 30
RUL_CAP     = 125
BATCH_SIZE  = 128
EPOCHS      = 50
LR          = 1e-3
RANDOM_SEED = 42
PROC_DIR    = ROOT / 'data' / 'processed'
SENSOR_COLS = [f's{i}' for i in [2,3,4,7,8,9,11,12,13,14,15,17,20,21]]

# Quantiles to predict simultaneously
QUANTILES   = [0.1, 0.5, 0.9]   # Q10, Q50, Q90
N_QUANTILES = len(QUANTILES)

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else
                      'mps'  if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Predicting quantiles: {QUANTILES}')

## 1. Load data

In [ ]:
train_df = pd.read_csv(PROC_DIR / 'train_features.csv')
test_df  = pd.read_csv(PROC_DIR / 'test_features.csv')

rmean_cols = [c for c in train_df.columns if '_rmean_' in c]
FEAT_COLS  = rmean_cols if len(rmean_cols) >= 10 else SENSOR_COLS
N_FEATURES = len(FEAT_COLS)

print(f'Train: {train_df.shape}  Test: {test_df.shape}')
print(f'Features: {N_FEATURES}')

## 2. Sliding window builder (same as T11)

In [ ]:
def build_windows(df, feat_cols, window_size=WINDOW_SIZE, is_test=False):
    X_list, y_list = [], []
    for _, grp in df.groupby('engine_id'):
        grp    = grp.sort_values('cycle')
        feats  = grp[feat_cols].values.astype(np.float32)
        labels = grp['RUL'].values.astype(np.float32)
        T      = len(feats)
        if T < window_size:
            pad    = np.tile(feats[0], (window_size - T, 1))
            feats  = np.vstack([pad, feats])
            labels = np.concatenate([np.full(window_size - T, labels[0]), labels])
            T      = window_size
        if is_test:
            X_list.append(feats[-window_size:])
            y_list.append(labels[-1])
        else:
            for i in range(window_size, T + 1):
                X_list.append(feats[i - window_size: i])
                y_list.append(labels[i - 1])
    return np.stack(X_list), np.array(y_list)


# Engine-level 80/20 split
all_eids   = train_df['engine_id'].unique()
np.random.shuffle(all_eids)
split_idx  = int(len(all_eids) * 0.8)
train_sub  = train_df[train_df['engine_id'].isin(all_eids[:split_idx])]
val_sub    = train_df[train_df['engine_id'].isin(all_eids[split_idx:])]

X_train, y_train = build_windows(train_sub, FEAT_COLS, is_test=False)
X_val,   y_val   = build_windows(val_sub,   FEAT_COLS, is_test=True)
X_test,  y_test  = build_windows(test_df,   FEAT_COLS, is_test=True)

print(f'X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}')

## 3. Dataset + DataLoader

In [ ]:
class RULDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):          return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

train_loader = DataLoader(RULDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(RULDataset(X_val,   y_val),   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(RULDataset(X_test,  y_test),  batch_size=BATCH_SIZE, shuffle=False)

## 4. Pinball Loss

The core difference from standard DL.

Pinball loss for quantile q:
- If actual > predicted (under-prediction): loss = q × |error|
- If actual < predicted (over-prediction):  loss = (1-q) × |error|

For q=0.9: under-prediction is penalised 9× more than over-prediction
→ model learns to predict high (conservative = safe for maintenance)

For q=0.1: over-prediction penalised 9× more
→ model learns to predict low (optimistic bound)

In [ ]:
class PinballLoss(nn.Module):
    """
    Pinball (quantile) loss for multi-quantile output.

    Args:
        quantiles: list of quantile levels, e.g. [0.1, 0.5, 0.9]

    Input:
        preds : (batch, n_quantiles)  — model output
        target: (batch,)              — true RUL

    Returns:
        scalar mean loss across batch and quantiles
    """
    def __init__(self, quantiles: list):
        super().__init__()
        # register as buffer so it moves to GPU with .to(device)
        self.register_buffer(
            'quantiles',
            torch.tensor(quantiles, dtype=torch.float32)
        )

    def forward(self, preds: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        # target: (B,) → (B, 1) for broadcasting across quantiles
        target  = target.unsqueeze(1)            # (B, 1)
        errors  = target - preds                 # (B, Q)  positive = under-predicted
        q       = self.quantiles.unsqueeze(0)    # (1, Q)

        # WHY max(q*e, (q-1)*e):
        # when e > 0 (under-predicted): loss = q * e     (penalise by quantile)
        # when e < 0 (over-predicted):  loss = (q-1) * e = (1-q) * |e|
        loss = torch.max(q * errors, (q - 1) * errors)  # (B, Q)
        return loss.mean()

## 5. Quantile Model Definitions

Each model is identical to T11 except the output head outputs N_QUANTILES neurons instead of 1.

In [ ]:
# ── Quantile MLP ──────────────────────────────────────────────────────
class QuantileMLP(nn.Module):
    """
    MLP — no recurrence, treats window as flat feature vector.
    WHY include: strongest non-temporal baseline.
    If MLP ≈ LSTM, the temporal structure isn't helping.
    """
    def __init__(self, n_features, window_size=WINDOW_SIZE,
                 hidden_size=128, n_quantiles=N_QUANTILES, dropout=0.2):
        super().__init__()
        input_dim  = n_features * window_size   # flatten entire window
        self.net   = nn.Sequential(
            nn.Linear(input_dim,   hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, n_quantiles),   # 3 outputs
        )

    def forward(self, x):
        # x: (B, W, F) → flatten → (B, W*F)
        x = x.view(x.size(0), -1)
        return self.net(x)   # (B, n_quantiles)


# ── Quantile RNN ──────────────────────────────────────────────────────
class QuantileRNN(nn.Module):
    def __init__(self, n_features, hidden_size=64,
                 n_layers=2, n_quantiles=N_QUANTILES, dropout=0.2):
        super().__init__()
        self.rnn = nn.RNN(
            input_size  = n_features,
            hidden_size = hidden_size,
            num_layers  = n_layers,
            batch_first = True,
            dropout     = dropout if n_layers > 1 else 0.0,
        )
        self.fc  = nn.Linear(hidden_size, n_quantiles)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])   # (B, n_quantiles)


# ── Quantile LSTM ─────────────────────────────────────────────────────
class QuantileLSTM(nn.Module):
    def __init__(self, n_features, hidden_size=64,
                 n_layers=2, n_quantiles=N_QUANTILES, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size  = n_features,
            hidden_size = hidden_size,
            num_layers  = n_layers,
            batch_first = True,
            dropout     = dropout if n_layers > 1 else 0.0,
        )
        self.fc   = nn.Linear(hidden_size, n_quantiles)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])   # (B, n_quantiles)


# ── Quantile GRU ──────────────────────────────────────────────────────
class QuantileGRU(nn.Module):
    def __init__(self, n_features, hidden_size=64,
                 n_layers=2, n_quantiles=N_QUANTILES, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(
            input_size  = n_features,
            hidden_size = hidden_size,
            num_layers  = n_layers,
            batch_first = True,
            dropout     = dropout if n_layers > 1 else 0.0,
        )
        self.fc  = nn.Linear(hidden_size, n_quantiles)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])   # (B, n_quantiles)


# ── Quantile Transformer ──────────────────────────────────────────────
class QuantileTransformer(nn.Module):
    def __init__(self, n_features, d_model=64, n_heads=4,
                 n_layers=2, n_quantiles=N_QUANTILES, dropout=0.1):
        super().__init__()
        self.input_proj  = nn.Linear(n_features, d_model)
        self.pos_enc     = nn.Embedding(WINDOW_SIZE, d_model)
        encoder_layer    = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.fc          = nn.Linear(d_model, n_quantiles)

    def forward(self, x):
        B, W, _   = x.shape
        positions = torch.arange(W, device=x.device).unsqueeze(0)
        x         = self.input_proj(x) + self.pos_enc(positions)
        x         = self.transformer(x)
        x         = x.mean(dim=1)        # mean pool
        return self.fc(x)                 # (B, n_quantiles)


print('Quantile model definitions ready.')
print(f'Output size per model: {N_QUANTILES} (Q10, Q50, Q90)')

## 6. Quantile Training Loop

In [ ]:
def train_quantile_model(
    model,
    train_loader,
    val_loader,
    quantiles=QUANTILES,
    epochs=EPOCHS,
    lr=LR,
    model_name='model',
    patience=10,
):
    """
    Train quantile model with pinball loss.
    Identical structure to T11 train_model() — only loss function differs.
    """
    model     = model.to(DEVICE)
    criterion = PinballLoss(quantiles).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=5, factor=0.5
    )

    best_val_loss = float('inf')
    best_weights  = None
    no_improve    = 0
    train_losses, val_losses = [], []

    for epoch in range(1, epochs + 1):

        # ── Train ─────────────────────────────────────────────────────
        model.train()
        batch_losses = []
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            preds = model(X_batch)              # (B, n_quantiles)
            loss  = criterion(preds, y_batch)   # pinball loss
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            batch_losses.append(loss.item())

        train_loss = np.mean(batch_losses)

        # ── Validate ──────────────────────────────────────────────────
        model.eval()
        val_batch_losses = []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(DEVICE)
                y_batch = y_batch.to(DEVICE)
                preds   = model(X_batch)
                loss    = criterion(preds, y_batch)
                val_batch_losses.append(loss.item())

        val_loss = np.mean(val_batch_losses)
        scheduler.step(val_loss)
        train_losses.append(train_loss)
        val_losses.append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights  = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve    = 0
        else:
            no_improve += 1

        if epoch % 10 == 0:
            print(f'  [{model_name}] Epoch {epoch:3d} | '
                  f'train={train_loss:.4f} | val={val_loss:.4f} | best={best_val_loss:.4f}')

        if no_improve >= patience:
            print(f'  [{model_name}] Early stop at epoch {epoch}')
            break

    model.load_state_dict(best_weights)
    return model, train_losses, val_losses

## 7. Quantile prediction + evaluation

In [ ]:
def predict_quantiles(
    model,
    test_loader,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Run inference and return all three quantile predictions.

    Returns:
        y_true : (n_engines,)
        q10    : (n_engines,)  — optimistic lower bound
        q50    : (n_engines,)  — median, used for RMSE/NASA
        q90    : (n_engines,)  — pessimistic upper bound
    """
    model.eval()
    all_preds, all_true = [], []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            preds = model(X_batch.to(DEVICE))           # (B, 3)
            preds = torch.clamp(preds, 0, RUL_CAP)     # clip all quantiles
            all_preds.extend(preds.cpu().numpy())
            all_true.extend(y_batch.numpy())

    preds_arr = np.array(all_preds)    # (n_engines, 3)
    y_true    = np.array(all_true)

    # WHY sort per-sample: model might output q10 > q50 occasionally
    # (no monotonicity constraint in output layer)
    # Sorting guarantees q10 <= q50 <= q90
    preds_arr = np.sort(preds_arr, axis=1)

    q10 = preds_arr[:, 0]
    q50 = preds_arr[:, 1]   # median — use for point prediction metrics
    q90 = preds_arr[:, 2]

    return y_true, q10, q50, q90


def evaluate_quantile_model(y_true, q10, q50, q90, model_name):
    """
    Evaluate on Q50 for RMSE/NASA (comparable to classical + T11).
    Also report interval coverage and average width.
    """
    print(f'\n=== {model_name} ===')

    # Point metrics on Q50
    results = evaluate(y_true, q50, model_name=f'{model_name} (Q50)')

    # Interval metrics
    interval_width    = np.mean(q90 - q10)
    # coverage: what % of true RULs fall within [Q10, Q90]?
    coverage          = np.mean((y_true >= q10) & (y_true <= q90)) * 100

    print(f'  Interval width (Q90-Q10) mean : {interval_width:.2f} cycles')
    print(f'  80% interval coverage         : {coverage:.1f}%  (target: ~80%)')

    return results, interval_width, coverage

## 8. Visualisation

In [ ]:
def plot_quantile_predictions(y_true, q10, q50, q90, model_name):
    """
    Two plots:
    1. Sorted predictions with Q10-Q90 shaded band — shows uncertainty
    2. Scatter Q50 vs true — standard point prediction view
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── Plot 1: sorted with interval band ────────────────────────────
    ax   = axes[0]
    sidx = np.argsort(y_true)

    smooth_q50 = pd.Series(q50[sidx]).rolling(10, center=True, min_periods=1).mean().values
    smooth_q10 = pd.Series(q10[sidx]).rolling(10, center=True, min_periods=1).mean().values
    smooth_q90 = pd.Series(q90[sidx]).rolling(10, center=True, min_periods=1).mean().values

    ax.plot(y_true[sidx],  color='steelblue',  lw=2,   label='True RUL')
    ax.plot(smooth_q50,    color='orange',      lw=2,   label='Q50 (median)')
    ax.fill_between(
        range(len(sidx)),
        smooth_q10, smooth_q90,
        color='orange', alpha=0.25,
        label='Q10–Q90 interval'
    )
    ax.set_xlabel('Engine (sorted by true RUL)')
    ax.set_ylabel('RUL')
    ax.set_title(f'{model_name} — Quantile Predictions')
    ax.legend()

    # ── Plot 2: scatter Q50 vs true ───────────────────────────────────
    ax = axes[1]
    ax.scatter(y_true, q50,  alpha=0.4, color='orange',    s=15,  label='Q50')
    ax.scatter(y_true, q10,  alpha=0.2, color='steelblue', s=8,   label='Q10')
    ax.scatter(y_true, q90,  alpha=0.2, color='red',       s=8,   label='Q90')
    ax.plot([0, 125], [0, 125], 'k--', lw=1.5, label='perfect')
    ax.set_xlabel('True RUL')
    ax.set_ylabel('Predicted RUL')
    ax.set_title(f'{model_name} — All Quantiles vs True')
    ax.legend(); ax.set_xlim(0, 130); ax.set_ylim(0, 130)

    plt.tight_layout()
    plt.savefig(f"{model_name.replace(' ', '_')}_quantile.png", dpi=150)
    plt.show()


def plot_loss_curves(train_losses, val_losses, model_name):
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.plot(train_losses, label='Train (pinball)', color='steelblue')
    ax.plot(val_losses,   label='Val (pinball)',   color='orange', ls='--')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Pinball Loss')
    ax.set_title(f'{model_name} — Training curve')
    ax.legend(); plt.tight_layout(); plt.show()

## 9. Train all five quantile models

In [ ]:
all_results  = {}
all_coverage = {}
all_widths   = {}

MODEL_CONFIGS = {
    'Q-MLP':         QuantileMLP(N_FEATURES),
    'Q-RNN':         QuantileRNN(N_FEATURES),
    'Q-LSTM':        QuantileLSTM(N_FEATURES),
    'Q-GRU':         QuantileGRU(N_FEATURES),
    'Q-Transformer': QuantileTransformer(N_FEATURES),
}

trained_models = {}

for name, model in MODEL_CONFIGS.items():
    print(f'\n{"="*50}')
    print(f'Training {name}...')
    print(f'{"="*50}')

    trained, t_losses, v_losses = train_quantile_model(
        model=model, train_loader=train_loader, val_loader=val_loader,
        model_name=name,
    )
    trained_models[name] = trained

    # Evaluate
    y_true, q10, q50, q90        = predict_quantiles(trained, test_loader)
    results, width, coverage     = evaluate_quantile_model(
        y_true, q10, q50, q90, model_name=name
    )
    all_results[name]  = results
    all_coverage[name] = coverage
    all_widths[name]   = width

    # Plots
    plot_loss_curves(t_losses, v_losses, model_name=name)
    plot_quantile_predictions(y_true, q10, q50, q90, model_name=name)

## 10. Final summary table — quantile models

In [ ]:
from src.evaluation.metrics import summarise_all_models

summary = summarise_all_models(all_results)

# Add interval metrics to summary
summary['interval_width'] = summary['model'].map(all_widths).round(2)
summary['coverage_pct']   = summary['model'].map(all_coverage).round(1)

print('\n=== QUANTILE MODELS SUMMARY ===')
print(summary.to_string())

## 11. Compare Q50 of quantile models vs classical + standard DL